# 02 - Training: V1/V2/V3 x 10 Seeds

Dieses Notebook wird **3x** ausgeführt (einmal pro Neck).
Zwischen den Runs nur `NECK = "..."` in Zelle 3 ändern.

Ablauf:
1. Setup (pip mit gepinnten Versionen, Drive, wandb)
2. Hardware-Check + `results/hardware.json`
3. Neck wählen (`fpn` / `aifi` / `mamba`)
4. Training über alle 10 Seeds (Skip/Resume-Logik)
5. Evaluation aller Seeds -> `results/{NECK}_seed_results.csv`
6. Validierung: 10/10 Seeds ok?
7. Konvergenz-Daten -> `results/{NECK}_convergence.csv`

In [ ]:
# Zelle 0 - Imports
import csv
import json
import os
import re
import subprocess
import sys
import time
from glob import glob
from pathlib import Path

import numpy as np
import torch

In [ ]:
# Zelle 1 - Setup
# Colab's vorinstalliertes torch/torchvision beibehalten.
# Nur fehlende Pakete nachinstallieren.
!pip uninstall -q -y torchaudio
!pip install -q mmengine "mmcv==2.1.0" mmdet
!pip install -q mamba-ssm causal-conv1d
!pip install -q pycocotools sahi wandb fvcore torchinfo deepdiff pandas

from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess

REPO = '/content/ba-mamba-neck'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/raphaelkach/ba-mamba-neck.git', REPO],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, REPO)

import wandb
wandb.login()

In [ ]:
# Zelle 2 - Hardware-Check
assert torch.cuda.is_available(), 'CUDA nicht verfügbar - A100-Runtime wählen!'

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
cuda_version = torch.version.cuda
pytorch_version = torch.__version__

import mmdet
mmdet_version = mmdet.__version__

try:
    import mamba_ssm
    mamba_version = mamba_ssm.__version__
except ImportError:
    raise RuntimeError(
        'mamba-ssm ist nicht importierbar! '
        'pip install mamba-ssm causal-conv1d ausführen.'
    )

hw = {
    'gpu_name': gpu_name,
    'vram_gb': vram_gb,
    'cuda_version': cuda_version,
    'pytorch_version': pytorch_version,
    'mmdet_version': mmdet_version,
    'mamba_ssm_version': mamba_version,
}

print(json.dumps(hw, indent=2))

if vram_gb < 38:
    print(f'WARNUNG: GPU hat nur {vram_gb} GB VRAM. '
          f'A100 (40/80 GB) wird empfohlen.')

Path('results').mkdir(exist_ok=True)
Path('results/hardware.json').write_text(json.dumps(hw, indent=2))
print('-> results/hardware.json gespeichert')

In [ ]:
# Zelle 3 - Config wählen
# Manuell ändern: "fpn", "aifi" oder "mamba"
NECK = "fpn"

In [ ]:
# Zelle 4 - Training über alle 10 Seeds (mit Resume-Logik)
# Resume-Logik bei Colab-Disconnect
# best_*.pth existiert -> skip (fertig)
# epoch_*.pth existiert -> --resume letzter Checkpoint
# nichts existiert -> Training von vorne


SEEDS = [42, 123, 456, 789, 1024, 2048, 3407, 4096, 5555, 7777]
DRIVE_BASE = f'/content/drive/MyDrive/ba/{NECK}'

for seed in SEEDS:
    work_dir = f'{DRIVE_BASE}/seed_{seed}'
    Path(work_dir).mkdir(parents=True, exist_ok=True)

 # --- Skip wenn best checkpoint bereits existiert ---
    if glob(f'{work_dir}/best_*.pth'):
        print(f'[{NECK}] Seed {seed} - best ckpt existiert, skip')
        continue

 # --- Resume wenn epoch checkpoint existiert (Colab-Disconnect) ---
    resume_flag = ''
    epoch_ckpts = sorted(glob(f'{work_dir}/epoch_*.pth'))
    if epoch_ckpts:
        last_ckpt = epoch_ckpts[-1]
        m = re.search(r'epoch_(\d+)', Path(last_ckpt).stem)
        resumed_epoch = int(m.group(1)) if m else '?'
        print(f'[{NECK}] Seed {seed} - RESUME ab epoch_{resumed_epoch} '
              f'({Path(last_ckpt).name})')
        resume_flag = f'--resume {last_ckpt}'
    else:
        print(f'\n{"="*60}')
        print(f'[{NECK}] Seed {seed} - Training von vorne')
        print(f'{"="*60}\n')

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    !PYTHONPATH=. python -m mmdet.utils.train \
        configs/{NECK}.py \
        --work-dir {work_dir} \
        --cfg-options randomness.seed={seed} \
        --cfg-options randomness.deterministic=True \
        {resume_flag}

    elapsed = time.time() - start_time
    peak_mem = torch.cuda.max_memory_allocated()

    run_meta = {
        'neck': NECK,
        'seed': seed,
        'train_time_sec': round(elapsed, 1),
        'train_time_h': round(elapsed / 3600, 2),
        'peak_gpu_mem_gb': round(peak_mem / 1e9, 2),
        'gpu_name': torch.cuda.get_device_name(0),
        'cuda_version': torch.version.cuda,
        'pytorch_version': torch.__version__,
        'mmdet_version': mmdet.__version__,
        'num_epochs': 24,
        'batch_size': 8,
        'lr': 0.0001,
        'optimizer': 'AdamW',
        'resumed': bool(epoch_ckpts),
    }
    meta_path = f'{work_dir}/run_meta.json'
    with open(meta_path, 'w') as f:
        json.dump(run_meta, f, indent=2)
    print(f'\n-> {meta_path} gespeichert')
    print(f'   Trainingszeit: {run_meta["train_time_h"]}h | '
          f'Peak GPU: {run_meta["peak_gpu_mem_gb"]} GB')

print(f'\n{"="*60}')
print(f'Alle Seeds für {NECK} abgeschlossen.')
print(f'{"="*60}')

In [ ]:
# Zelle 5 - Ergebnisse sammeln (Eval best checkpoints)
import necks  # register custom necks

from mmdet.apis import init_detector
from mmdet.evaluation import CocoMetric
from mmengine.config import Config
from mmengine.runner import Runner

SEEDS = [42, 123, 456, 789, 1024, 2048, 3407, 4096, 5555, 7777]
DRIVE_BASE = f'/content/drive/MyDrive/ba/{NECK}'

# VisDrone 10 classes (same order as metainfo in dataset config)
CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor',
]

# CSV header
header = [
    'neck', 'seed',
    'mAP', 'mAP_50', 'mAP_75', 'AP_S', 'AP_M', 'AP_L',
    'AR_1', 'AR_10', 'AR_100', 'AR_S', 'AR_M', 'AR_L',
]
header += [f'AP_{c}' for c in CLASS_NAMES]
header += [f'AR_{c}' for c in CLASS_NAMES]
header += ['best_epoch', 'train_time_h', 'peak_gpu_mem_gb']

rows = []
for seed in SEEDS:
    work_dir = Path(f'{DRIVE_BASE}/seed_{seed}')
    ckpt_candidates = sorted(glob(str(work_dir / 'best_*.pth')))
    if not ckpt_candidates:
        print(f'[{NECK}] Seed {seed} - kein best checkpoint gefunden, skip')
        continue
    ckpt = ckpt_candidates[0]

 # Best epoch number from filename (e.g. best_coco_bbox_mAP_epoch_18.pth)
    m = re.search(r'epoch_(\d+)', Path(ckpt).stem)
    best_epoch = int(m.group(1)) if m else -1

 # Run meta
    meta_path = work_dir / 'run_meta.json'
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}

 # Build runner from config for evaluation
    cfg = Config.fromfile(f'configs/{NECK}.py')
    cfg.work_dir = str(work_dir)
    cfg.load_from = ckpt

    runner = Runner.from_cfg(cfg)
    metrics = runner.test()

 # metrics is a dict with keys like 'coco/bbox_mAP', 'coco/bbox_mAP_50', ...
    def g(key, default=0.0):
        return round(metrics.get(f'coco/{key}', default), 4)

    row = {
        'neck': NECK,
        'seed': seed,
        'mAP': g('bbox_mAP'),
        'mAP_50': g('bbox_mAP_50'),
        'mAP_75': g('bbox_mAP_75'),
        'AP_S': g('bbox_mAP_s'),
        'AP_M': g('bbox_mAP_m'),
        'AP_L': g('bbox_mAP_l'),
        'AR_1': g('bbox_mAR_1'),
        'AR_10': g('bbox_mAR_10'),
        'AR_100': g('bbox_mAR_100'),
        'AR_S': g('bbox_mAR_s'),
        'AR_M': g('bbox_mAR_m'),
        'AR_L': g('bbox_mAR_l'),
    }

 # Per-class AP and AR
    for cls in CLASS_NAMES:
        row[f'AP_{cls}'] = g(f'bbox_{cls}_precision', 0.0)
        row[f'AR_{cls}'] = g(f'bbox_{cls}_recall', 0.0)

    row['best_epoch'] = best_epoch
    row['train_time_h'] = meta.get('train_time_h', -1)
    row['peak_gpu_mem_gb'] = meta.get('peak_gpu_mem_gb', -1)

    rows.append(row)
    print(f'Seed {seed}: mAP={row["mAP"]:.4f} AP_S={row["AP_S"]:.4f} '
          f'AP_M={row["AP_M"]:.4f} AP_L={row["AP_L"]:.4f} '
          f'best_epoch={best_epoch}')

# Write CSV
csv_path = Path(f'results/{NECK}_seed_results.csv')
csv_path.parent.mkdir(exist_ok=True)
with csv_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print(f'\n-> {csv_path} ({len(rows)} seeds)')

# Summary: Mean +/- Std, Median
if rows:
    for metric in ['mAP', 'AP_S', 'AP_M', 'AP_L', 'mAP_50', 'mAP_75']:
        vals = np.array([r[metric] for r in rows])
        print(f'  {metric:>8}: mean={vals.mean():.4f} +/- {vals.std():.4f}  '
              f'median={np.median(vals):.4f}')

In [ ]:
# Zelle 6 - Validierung vor Neck-Wechsel
assert len(rows) == 10, (
    f'Nur {len(rows)}/10 Seeds fertig! '
    f'Fehlende Seeds müssen erst trainiert werden.'
)

for r in rows:
    assert r['mAP'] > 0.01, (
        f"Seed {r['seed']} mAP={r['mAP']:.4f} verdächtig niedrig - "
        f"Training prüfen!"
    )
    assert r['train_time_h'] > 0, (
        f"Seed {r['seed']} Trainingszeit fehlt (train_time_h={r['train_time_h']})"
    )

print(f'Alle 10 Seeds für {NECK} validiert — bereit für nächsten Neck.')

In [ ]:
# Zelle 7 - Konvergenz-Daten aus mmdet JSON-Logs
SEEDS = [42, 123, 456, 789, 1024, 2048, 3407, 4096, 5555, 7777]
DRIVE_BASE = f'/content/drive/MyDrive/ba/{NECK}'

conv_header = [
    'neck', 'seed', 'epoch',
    'train_loss', 'cls_loss', 'bbox_loss', 'centerness_loss',
    'val_mAP', 'val_AP_S', 'val_AP_M', 'val_AP_L',
    'lr',
]

conv_rows = []
for seed in SEEDS:
    work_dir = Path(f'{DRIVE_BASE}/seed_{seed}')

 # Find the JSON log file (mmengine writes vis_data/<timestamp>/vis_data/scalars.json
 # or a flat <timestamp>.log.json in work_dir)
    log_files = sorted(glob(str(work_dir / '*.log.json')))
    if not log_files:
 # fallback: vis_data scalars
        log_files = sorted(glob(str(work_dir / 'vis_data' / '*.json')))
    if not log_files:
        print(f'[{NECK}] Seed {seed} - kein Log gefunden, skip')
        continue

    log_path = Path(log_files[-1])  # latest log

 # Parse mmengine JSON log: each line is a JSON dict with
 # mode='train'/'val', epoch, iter, lr, loss, ...
    train_epochs = {}  # epoch -> aggregated losses
    val_epochs = {}    # epoch -> metrics

    for line in log_path.read_text().strip().splitlines():
        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            continue

        mode = entry.get('mode')
        epoch = entry.get('epoch')
        if epoch is None:
            continue

        if mode == 'train':
 # Accumulate last logged values per epoch
            train_epochs[epoch] = {
                'train_loss': entry.get('loss', 0.0),
                'cls_loss': entry.get('loss_cls', 0.0),
                'bbox_loss': entry.get('loss_bbox', 0.0),
                'centerness_loss': entry.get('loss_centerness', 0.0),
                'lr': entry.get('lr', 0.0),
            }
        elif mode == 'val':
            val_epochs[epoch] = {
                'val_mAP': entry.get('coco/bbox_mAP', 0.0),
                'val_AP_S': entry.get('coco/bbox_mAP_s', 0.0),
                'val_AP_M': entry.get('coco/bbox_mAP_m', 0.0),
                'val_AP_L': entry.get('coco/bbox_mAP_l', 0.0),
            }

    all_epochs = sorted(set(train_epochs) | set(val_epochs))
    for ep in all_epochs:
        row = {'neck': NECK, 'seed': seed, 'epoch': ep}
        row.update(train_epochs.get(ep, {
            'train_loss': None, 'cls_loss': None,
            'bbox_loss': None, 'centerness_loss': None, 'lr': None,
        }))
        row.update(val_epochs.get(ep, {
            'val_mAP': None, 'val_AP_S': None,
            'val_AP_M': None, 'val_AP_L': None,
        }))
        conv_rows.append(row)

    print(f'[{NECK}] Seed {seed}: {len(all_epochs)} Epochen aus {log_path.name}')

# Write convergence CSV
conv_path = Path(f'results/{NECK}_convergence.csv')
with conv_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=conv_header)
    writer.writeheader()
    writer.writerows(conv_rows)

print(f'\n-> {conv_path} ({len(conv_rows)} Zeilen, {len(SEEDS)} Seeds)')